# Quant Research Engine - Colab Validation

Runs the full research pipeline on real SPY/QQQ data
using the updated high-threshold configuration.

Expected runtime: 15-25 min on Colab CPU.

Tests:
- Full pipeline end-to-end
- GBM n_estimators=200 with high thresholds [0.65..0.90]
- Hold bars [10, 15, 20, 30]
- All 14 promotion gates
- Cost stress (0, 2.5, 5, 10, 20 bps)
- Delay stress (0, 1, 2, 3 bars)
- Bootstrap confidence
- Placebo percentile

Output: artifacts_colab/ directory with full results.\n**Updated:** includes 9 new pv-2.2.0 signal-extension features\n(overnight gap, intraday return, range position, RSI-14, Bollinger, 52-week position,\nvol-of-vol, price-volume corr, Amihud illiquidity). Run ID: artifacts_colab.

In [1]:
import os, sys, subprocess
from pathlib import Path

repo_path = Path('/content/ml-2')
if not repo_path.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ProgrammerMarnus/ML-2.git', str(repo_path)], check=True)
os.chdir(repo_path)
sys.path.insert(0, str(repo_path))

print(f'Repo: {repo_path}')
result = subprocess.run(['git', 'branch', '--show-current'], capture_output=True, text=True, cwd=repo_path)
print(f'Branch: {result.stdout.strip()}')
print(f'Files: {len(list(repo_path.rglob('*')))}')


Repo: /content/ml-2
Branch: main
Files: 472


In [2]:
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], capture_output=True, text=True)
if result.returncode != 0:
    print('INSTALL FAILED:')
    print(result.stderr)
else:
    print('Installed OK')


Installed OK


In [ ]:
import json, pandas as pd
import sys
from pathlib import Path

# Assuming repo_path is available from previous cells (defined as Path('/content/ml-2')).
# Add the 'src' directory of the repository to sys.path.
# This is a common pattern for Python packages structured with a 'src' directory
# and resolves ModuleNotFoundError if the package is under 'src'.
sys.path.insert(0, str(repo_path / 'src'))

from quant_research.run import run_research_pipeline
from quant_research.config import load_config

cfg = load_config('configs/real_spy.yaml')
print('Config loaded:')
print(f'  Model: {cfg.model.type} (n_est={cfg.model.gb_n_estimators}, lr={cfg.model.gb_learning_rate})')
print(f'  Thresholds: {cfg.research.threshold_candidates}')
print(f'  Hold bars: {cfg.research.hold_candidates}')
print(f'  Data: {cfg.data.assets} ({cfg.data.start} to {cfg.data.end})')

output_dir = 'artifacts_colab'
from quant_research.features.price_volume import SIGNAL_EXTENSION_COLUMNS
print(f'Signal extensions (pv-2.2.0): {len(SIGNAL_EXTENSION_COLUMNS)} features')
print(f'Running pipeline (output: {output_dir})...')
print('(This will take 15-25 minutes)')

report = run_research_pipeline(cfg, output_dir)
print('Pipeline complete!')

from quant_research.features.price_volume import SIGNAL_EXTENSION_COLUMNS
print(f'Signal extensions (pv-2.2.0): {len(SIGNAL_EXTENSION_COLUMNS)} features')
print(f'  {SIGNAL_EXTENSION_COLUMNS}')
print(f'Total features: {len(report["experiment_record"]["features"]) if "features" in report["experiment_record"] else "see manifest"}')

In [ ]:
print('='*70)
print('RESULTS SUMMARY')
print('='*70)
summary = report['baseline_summary']
print(f'Performance:')
print(f'  Mean OOS Sharpe:   {summary["mean_oos_sharpe"]:+.4f}')
print(f'  Median OOS Sharpe: {summary["median_oos_sharpe"]:+.4f}')
print(f'  Full Net Sharpe:   {summary["full_oos_net_sharpe"]:+.4f}')
print(f'  Full Gross Sharpe: {summary["full_oos_gross_sharpe"]:+.4f}')
print(f'  AUC:               {summary["mean_oos_auc"]:.4f}')
print(f'  Max DD:            {summary["full_oos_max_dd"]:.4f}')
print(f'  Trades:            {summary["total_oos_trades"]}')
print(f'  Positive Folds:    {summary["positive_folds"]} / {summary["n_folds"]}')
promo = report['promotion']
print(f'Promotion:')
print(f'  State:   {promo["state"]}')
print(f'  Passed:  {len(promo["passed_gates"])} gates')
for g in promo['passed_gates']:
    print(f'    OK   {g}')
print(f'  Failed:  {len(promo["failed_gates"])} gates')
for g in promo['failed_gates']:
    print(f'    FAIL {g}')
boot = report['bootstrap']
print(f'Bootstrap:')
print(f'  Mean:       {boot["mean"]:+.4f}')
print(f'  95% CI:     [{boot["lo"]:+.4f}, {boot["hi"]:+.4f}]')
print(f'  P(Sharpe>0): {boot["positive_prob"]:.4f}')
pl = report['placebo_statistics']
print(f'Placebo:')
print(f'  Percentile:  {pl["percentile"]:.4f}')
print(f'  Adj p-value: {pl["adjusted_p"]:.4f}')
print(f'  Null Mean:   {pl["null_mean"]:+.4f}')
print(f'  Observed:    {pl["observed"]:+.4f}')
print(f'  N Runs:      {pl["n_runs"]}')

# Safely extract and convert cost_stress scenarios to records
cost_scenarios = report.get('cost_stress', {})
if isinstance(cost_scenarios, dict):
    cost_scenarios = cost_scenarios.get('scenarios', cost_scenarios.get('results', []))
if isinstance(cost_scenarios, pd.DataFrame):
    cost_scenarios = cost_scenarios.to_dict(orient='records')

print(f'Cost Stress:')
for r in cost_scenarios:
    status = 'OK' if r['sharpe'] > 0 else 'FAIL'
    print(f'  fee={r["fee_bps"]:5.1f} bps: sharpe={r["sharpe"]:+.4f} [{status}]')

# Safely extract and convert delay_stress scenarios to records
delay_scenarios = report.get('delay_stress', {})
if isinstance(delay_scenarios, dict):
    delay_scenarios = delay_scenarios.get('scenarios', delay_scenarios.get('results', []))
if isinstance(delay_scenarios, pd.DataFrame):
    delay_scenarios = delay_scenarios.to_dict(orient='records')

print(f'Delay Stress:')
for r in delay_scenarios:
    status = 'OK' if r['sharpe'] > 0 else 'FAIL'
    print(f'  delay={r["delay_bars"]:2d} bars: sharpe={r["sharpe"]:+.4f} [{status}]')

print(f'Evidence: {report["experiment_record"]["evidence_status"]}')
print(f'Experiment ID: {report["experiment_record"]["experiment_id"]}')
print(f'Manifest: {report["manifest_path"]}')

In [ ]:
print('='*70)
print('PER-FOLD RESULTS')
print('='*70)
import glob
folds_csv = pd.read_csv(glob.glob(f'{output_dir}/*_folds.csv')[0])
for _, r in folds_csv.iterrows():
    print(f'\nFold {int(r["fold_id"]):2d} ({r["test_start"]} -> {r["test_end"]}):')
    print(f'  Threshold:  {r["threshold"]:.2f}')
    print(f'  Model:      {r["model_type"]}')
    print(f'  Hold:       {r.get("hold_bars", "N/A")}')
    print(f'  Features:   {r.get("selected_features", "N/A")[:60]}...')
    print(f'  OOS Sharpe: {r["oos_sharpe"]:+.4f}')
    print(f'  OOS Max DD: {r["oos_max_dd"]:.4f}')
    print(f'  Trades:     {r["oos_trades"]}')
    print(f'  Turnover:   {r["oos_turnover"]:.2f}')
    print(f'  Net Return: {r["oos_net_return"]:+.4f}')
    print(f'  Gross Ret:  {r["oos_gross_return"]:+.4f}')
    print(f'  AUC:        {r["oos_auc"]:.4f}')
    print(f'  Brier:      {r["oos_brier"]:.4f}')


## Output

The notebook produces:
- Full pipeline results JSON
- Per-fold CSV
- Experiment registry (append-only)
- Reproducibility manifest
- Test lock (for locked-test protocol)
- Trial counter (persistent)

Download the `artifacts_colab/` directory from Colab and share it.
I will analyze the results and update DISCOVERY_ANALYSIS.txt.